# Colab Worker B: Track B - Classical Spread-Spectrum Band Sweep
This notebook runs on **Colab Account 2** (T4/V100/A100 GPU).
- 32-bit blind spread-spectrum watermark embedder & correlation decoder
- Sweeps 8 frequency bands × strengths × 4 PSNR targets (35, 38, 41, 44 dB)
- Logs Bit Accuracy, SSIM, and detection TPR at FPR=1%
- Writes results to `results/track_b_sweep.csv`

In [ ]:
# 1. Check GPU and mount Google Drive
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')
SHARED_DIR = '/content/drive/MyDrive/watermark_internship'
!mkdir -p $SHARED_DIR/results $SHARED_DIR/checkpoints

In [ ]:
# 2. Clone repository & install dependencies
!rm -rf /content/repo
!git clone https://github.com/ashhyyyy-vis/Internship.git /content/repo
%cd /content/repo
!pip install -q --no-warn-conflicts diffusers transformers accelerate "compressai>=1.2.4" kornia lpips pandas scipy tqdm

In [ ]:
# 3. Setup Dataset Splits (Drive unpack with automatic download fallback)
import os
!mkdir -p /content/repo/data/processed

# Unpack from Google Drive if processed_splits.zip exists
!if [ -f $SHARED_DIR/processed_splits.zip ]; then \
    echo "Found processed_splits.zip in Google Drive. Extracting..."; \
    unzip -q -o $SHARED_DIR/processed_splits.zip -d /content/repo/data/; \
    unzip -q -o $SHARED_DIR/processed_splits.zip -d /content/repo/data/processed/; \
  fi

# Fallback: if data/fit is still missing, automatically download and prepare in Colab (<30s)
if not os.path.exists('/content/repo/data/fit') and not os.path.exists('/content/repo/data/processed/fit'):
    print("Dataset splits not found in Drive. Preparing dataset directly in Colab...")
    !python data_prep.py
    !cp /content/repo/data/processed_splits.zip $SHARED_DIR/ 2>/dev/null || true

# Mirror directory structure so any path works seamlessly
!mkdir -p /content/repo/data/processed/fit /content/repo/data/processed/test /content/repo/data/processed/train
!cp -rn /content/repo/data/fit/* /content/repo/data/processed/fit/ 2>/dev/null || true
!cp -rn /content/repo/data/test/* /content/repo/data/processed/test/ 2>/dev/null || true
!cp -rn /content/repo/data/train/* /content/repo/data/processed/train/ 2>/dev/null || true

n_fit = len(os.listdir('/content/repo/data/processed/fit')) if os.path.exists('/content/repo/data/processed/fit') else 0
print(f"✓ Dataset ready! Found {n_fit} fit images in data/processed/fit/")
assert n_fit >= 300, f"Expected at least 300 fit images, found {n_fit}"

In [ ]:
# 4. Run Classical Spread-Spectrum Sweep
!python -m track_b.sweep \
    --manifest data/manifests/fit_manifest.json \
    --images_dir data/processed/fit \
    --out results/track_b_sweep.csv \
    --attacks compressai sd_vae \
    --limit 300

!cp results/track_b_sweep.csv $SHARED_DIR/results/
print('✓ Track B sweep complete! Saved to Drive.')